# Analysing a Solana account

Sync one devnet account into a temporary database, then look at its activity with pandas.

In [ ]:
import tempfile
from pathlib import Path

from multichain_indexer import Indexer

ACCOUNT = "FwcsNmTLbf8VVqxuBMjEDTh39gXR87n3vAnAwvAiPtRy"
folder = tempfile.TemporaryDirectory()
indexer = Indexer("solana", database=Path(folder.name) / "example.db")
indexer.sync(ACCOUNT)

Exports carry the shared fields as columns and the `native` block as JSON. Priority fees are in lamports for every transaction version, so they can be compared directly.

In [ ]:
import io
import json

import pandas as pd

frame = pd.read_csv(io.StringIO(indexer.export("csv", account=ACCOUNT)))
native = frame["native"].map(json.loads)
frame["version"] = native.map(lambda n: n["version"])
frame["kind"] = native.map(lambda n: n["kind"])
frame["priority_fee"] = native.map(lambda n: n["priority_fee_lamports"])
frame[["time", "version", "kind", "status", "fee", "priority_fee"]].head()

In [ ]:
frame.groupby("version")[["fee", "priority_fee"]].agg(["count", "sum", "max"])

In [ ]:
indexer.metrics(days=365, account=ACCOUNT).native

In [ ]:
indexer.close()
folder.cleanup()